# 🏗️ Fabric Environment Setup Notebook
This notebook creates a bronze and silver Lakehouse, EDW warehouse, Fabric SQL Database, and various connections. \
**After running this notebook, manually upload `pipeline_activities.json` to the `Files/` folder in the Lakehouse.**

## 📌 Parameters

In [ ]:
# Workspace and Artifacts
workspace_name = "Data Engineering [DEV]"  # 👈 Set your workspace name here
bronze_lakehouse_name = "Bronze_Lake"
silver_lakehouse_name = "Silver_Lake"
warehouse_name = "Gold_EDW"
sql_db_name = "UTIL_Database"

# 🔌 Set Parameters for required connections
# ADLS Gen2
storage_url = "https://<your-storage-account>.dfs.core.windows.net/<your-container>"
adls_connection_name = "ADLSConn"

# Azure SQL
azure_sql_conn_name = "AzureSQLConn"
azure_sql_server = "<your-server>.database.windows.net"
azure_sql_db = "<your-db-name>"
azure_sql_user = "<username>"
azure_sql_pass = "<password>"

# SQL Server (on-prem via Gateway)
sql_server_conn_name = "OnPremSQLConn"
sql_server_name = "<sql-server-host>"
sql_server_db = "<sql-db-name>"
sql_server_user = "<username>"
sql_server_pass = "<password>"
gateway_id = "<your-gateway-id>"

# ODBC
odbc_conn_name = "OdbcConn"
odbc_conn_string = "DSN=MyOdbcSource;UID=user;PWD=pass"

# Dynamics 365 (CRM)
dynamics_crm_conn_name = "DynamicsCrmConn"
dynamics_crm_url = "https://<org>.crm.dynamics.com"
dynamics_crm_client_id = "<client-id>"
dynamics_crm_client_secret = "<client-secret>"
dynamics_crm_tenant_id = "<tenant-id>"

# Business Central
bc_conn_name = "BusinessCentralConn"
bc_url = "https://api.businesscentral.dynamics.com/v2.0/<tenant-id>/sandbox/ODataV4/"
bc_client_id = "<client-id>"
bc_client_secret = "<client-secret>"
bc_tenant_id = "<tenant-id>"

In [ ]:
# 🔍 Get workspace ID from workspace name using sempy_labs
import sempy_labs.admin as admin

workspace_df = admin.list_workspaces(workspace=workspace_name)
if workspace_df.empty:
    raise ValueError(f"No workspace found with name '{workspace_name}'")
workspace_id = workspace_df.iloc[0]['Id']
print("Workspace ID:", workspace_id)

In [ ]:
# 🔐 Authenticate using Fabric's built-in identity
import notebookutils
import requests

fmd_api_access_token = notebookutils.credentials.getToken('https://analysis.windows.net/powerbi/api')
if not fmd_api_access_token:
    raise Exception("Failed to get access token")

def create_fabric_session(fabric_token):
    fabric_headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {fabric_token}'
    }
    session = requests.Session()
    session.headers.update(fabric_headers)
    return session

session = create_fabric_session(fmd_api_access_token)

def artifact_exists(url, name):
    response = session.get(url)
    if response.status_code == 200:
        items = response.json().get('value', [])
        return any(item.get('name') == name for item in items)
    return False

## ✅ Create Fabric Artifacts

In [ ]:
# ✅ Create Bronze Lakehouse
lakehouse_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses"
if not artifact_exists(lakehouse_url, bronze_lakehouse_name):
    payload = {"name": bronze_lakehouse_name, "type": "Lakehouse", "displayName": bronze_lakehouse_name}
    resp = session.post(lakehouse_url, json=payload)
    print("Lakehouse created:", resp.status_code, resp.text)
else:
    print("Lakehouse already exists.")

In [ ]:
# ✅ Create Silver Lakehouse
lakehouse_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses"
if not artifact_exists(lakehouse_url, silver_lakehouse_name):
    payload = {"name": silver_lakehouse_name, "type": "Lakehouse", "displayName": silver_lakehouse_name}
    resp = session.post(lakehouse_url, json=payload)
    print("Lakehouse created:", resp.status_code, resp.text)
else:
    print("Lakehouse already exists.")

In [ ]:
# ✅ Create Fabric EDW Warhouse
warehouse_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/warehouses"
if not artifact_exists(warehouse_url, warehouse_name):
    payload = {"name": warehouse_name, "type": "Warehouse", "displayName": warehouse_name}
    resp = session.post(warehouse_url, json=payload)
    print("EDW Warehouse created:", resp.status_code, resp.text)
else:
    print("EDW Warehouse already exists.")

In [ ]:
# ✅ Create Fabric SQL Database
sql_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items"
if not artifact_exists(sql_url, sql_db_name):
    payload = {"name": sql_db_name, "type": "SQLDatabase", "displayName": sql_db_name}
    resp = session.post(sql_url, json=payload)
    print("SQL Database created:", resp.status_code, resp.text)
else:
    print("SQL Database already exists.")

## 🔗 Create Data Connections

In [ ]:
# ✅ Set general connection url
conn_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/connections"

In [ ]:
# 🔌 Create ADLS Gen2 Connection
adls_payload = {
    "name": adls_connection_name,
    "type": "AzureDataLakeStorageGen2",
    "properties": {
        "url": storage_url
    }
}
if not artifact_exists(conn_url, adls_connection_name):
    resp = session.post(conn_url, json=adls_payload)
    print("Connection created:", resp.status_code, resp.text)
else:
    print("Connection already exists.")

In [ ]:
# 🔌 Azure SQL Connection
azure_sql_payload = {
    "name": azure_sql_conn_name,
    "type": "SqlServer",
    "properties": {
        "server": azure_sql_server,
        "database": azure_sql_db,
        "authenticationKind": "UsernamePassword",
        "username": azure_sql_user,
        "password": azure_sql_pass
    }
}
if not artifact_exists(conn_url, azure_sql_conn_name):
    resp = session.post(conn_url, json=azure_sql_payload)
    print("Azure SQL connection created:", resp.status_code, resp.text)
else:
    print("Azure SQL connection already exists.")

In [ ]:
# 🔌 SQL Server (On-Prem) Connection via Gateway
sql_server_payload = {
    "name": sql_server_conn_name,
    "type": "SqlServer",
    "properties": {
        "server": sql_server_name,
        "database": sql_server_db,
        "authenticationKind": "UsernamePassword",
        "username": sql_server_user,
        "password": sql_server_pass,
        "gatewayId": gateway_id
    }
}
if not artifact_exists(conn_url, sql_server_conn_name):
    resp = session.post(conn_url, json=sql_server_payload)
    print("SQL Server connection created:", resp.status_code, resp.text)
else:
    print("SQL Server connection already exists.")

In [ ]:
# 🔌 ODBC Connection
odbc_payload = {
    "name": odbc_conn_name,
    "type": "Odbc",
    "properties": {
        "connectionString": odbc_conn_string
    }
}
if not artifact_exists(conn_url, odbc_conn_name):
    resp = session.post(conn_url, json=odbc_payload)
    print("ODBC connection created:", resp.status_code, resp.text)
else:
    print("ODBC connection already exists.")

In [ ]:
# 🔌 Dynamics 365 CRM Connection
crm_payload = {
    "name": dynamics_crm_conn_name,
    "type": "DynamicsCrm",
    "properties": {
        "url": dynamics_crm_url,
        "authenticationKind": "OAuth2",
        "clientId": dynamics_crm_client_id,
        "clientSecret": dynamics_crm_client_secret,
        "tenantId": dynamics_crm_tenant_id
    }
}
if not artifact_exists(conn_url, dynamics_crm_conn_name):
    resp = session.post(conn_url, json=crm_payload)
    print("Dynamics CRM connection created:", resp.status_code, resp.text)
else:
    print("Dynamics CRM connection already exists.")

In [ ]:
# 🔌 Business Central Connection
bc_payload = {
    "name": bc_conn_name,
    "type": "BusinessCentral",
    "properties": {
        "url": bc_url,
        "authenticationKind": "OAuth2",
        "clientId": bc_client_id,
        "clientSecret": bc_client_secret,
        "tenantId": bc_tenant_id
    }
}
if not artifact_exists(conn_url, bc_conn_name):
    resp = session.post(conn_url, json=bc_payload)
    print("Business Central connection created:", resp.status_code, resp.text)
else:
    print("Business Central connection already exists.")

✅ **Next Step:**
- Go to your Fabric Lakehouse (`Bronze_Data`)
- Open the **Files** folder
- Upload `pipeline_activities.json` there
- Then proceed to run the next notebook to build the pipeline